# Setup dan Import Library

In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import faiss
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

import sys
sys.path.insert(0, os.path.abspath('..'))
from src.retrieval.fusion import reciprocal_rank_fusion, get_top_k
from src.retrieval.evaluation import (
    ndcg_at_k, mean_reciprocal_rank, recall_at_k,
    mean_average_precision, cohens_kappa, evaluate_all_metrics
)
from src.retrieval.preprocessing import (
    create_stemmer, create_stopword_remover,
    clean_text_semantic, clean_text_lexical
)
from src.retrieval.retrieval import HybridRetriever 

INDEX_DIR = '../index'
EVAL_DIR  = '../data'
print("✅ Import selesai")

# Load Index

In [ ]:
df               = pd.read_pickle(f'{INDEX_DIR}/dataset_processed.pkl')
tfidf_data       = pickle.load(open(f'{INDEX_DIR}/tfidf.pkl', 'rb'))
bm25             = pickle.load(open(f'{INDEX_DIR}/bm25.pkl', 'rb'))
faiss_index      = faiss.read_index(f'{INDEX_DIR}/dense.faiss')
sbert_model      = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
stemmer          = create_stemmer()
stopword_remover = create_stopword_remover()

retriever = HybridRetriever(
    tfidf_data['vectorizer'], tfidf_data['matrix'],
    bm25, faiss_index, sbert_model, stemmer, stopword_remover
)
print("✅ Retriever siap")

# Definisi 30 Query Uji

In [ ]:
TEST_QUERIES = [
    # Leksikal (nama spesifik)
    "Taman Nasional Komodo",
    "Candi Borobudur Magelang",
    "Pantai Kuta Bali",
    "Kebun Raya Bogor",
    "museum batik Yogyakarta",
    "Taman Mini Indonesia Indah Jakarta",
    "Kawah Ijen Banyuwangi",
    "Gunung Bromo Jawa Timur",

    # Semantik (parafrase / konseptual)
    "tempat menenangkan pikiran dengan alam",
    "wisata alam yang cocok untuk meditasi",
    "destinasi petualangan outdoor seru",
    "liburan romantis pemandangan alam indah",
    "tempat belajar sejarah budaya Jawa",
    "wisata edukasi untuk anak-anak",

    # Campuran (kontekstual)
    "pantai untuk snorkeling melihat terumbu karang",
    "gunung untuk pendakian pemula",
    "tempat belanja oleh-oleh khas daerah",
    "taman hiburan seru untuk keluarga",
    "air terjun tersembunyi",
    "wisata religi tempat ibadah",
    "kuliner dan wisata di Bandung",
    "cagar alam hutan hujan tropis",
    "wisata murah meriah untuk backpacker",
    "resort dan resort alam mewah",
    "spot foto aesthetic kekinian",
    "kegiatan water sport di laut",
    "wisata malam kota Jakarta",
    "tempat camping dengan pemandangan bintang",
    "danau vulkanik eksotis",
    "wisata sejarah perang kemerdekaan",
]

print(f"Total query uji: {len(TEST_QUERIES)}")
for i, q in enumerate(TEST_QUERIES, 1):
    print(f"  {i:2d}. {q}")

# Jalankan Retrieval untuk Semua Query

In [ ]:
from tqdm import tqdm

results_all = {}   # {query: {'hybrid': [...], 'tfidf': [...], 'bm25': [...], 'dense': [...]}}

for query in tqdm(TEST_QUERIES, desc="Retrieving"):
    res = retriever.hybrid(query, top_k=10)
    results_all[query] = res

# Simpan hasil retrieval (agar tidak perlu diulang)
with open(f'{EVAL_DIR}/retrieval_results.json', 'w') as f:
    json.dump(results_all, f, indent=2)

print(f"✅ Retrieval selesai untuk {len(TEST_QUERIES)} query")
print(f"💾 Tersimpan di {EVAL_DIR}/retrieval_results.json")

# Inference Anotasi Manual

In [ ]:
# ============================================================
# Setiap anggota kelompok menjalankan cell ini dan
# menilai relevansi dokumen (1=relevan, 0=tidak relevan)
#
# PETUNJUK:
# - Jalankan cell ini
# - Untuk setiap dokumen, klik "Relevan" atau "Tidak Relevan"
# - Anotasi tersimpan otomatis setelah setiap query
# ============================================================

ANNOTATOR_NAME = "annotator_1"   # ← ganti untuk setiap anggota kelompok
ANNOTATION_FILE = f'{EVAL_DIR}/annotations_{ANNOTATOR_NAME}.json'
METHOD_TO_ANNOTATE = 'hybrid'    # Anotasi hasil hybrid (atau bisa 'tfidf'/'bm25'/'dense')

# Load existing annotations jika ada
if os.path.exists(ANNOTATION_FILE):
    with open(ANNOTATION_FILE, 'r') as f:
        annotations = json.load(f)
    print(f"📂 Melanjutkan anotasi yang sudah ada: {len(annotations)}/{len(TEST_QUERIES)} query")
else:
    annotations = {}
    print("📝 Memulai anotasi baru")

# Filter query yang belum dianotasi
remaining_queries = [q for q in TEST_QUERIES if q not in annotations]
print(f"📋 Sisa query untuk dianotasi: {len(remaining_queries)}")

if not remaining_queries:
    print("🎉 Semua query sudah dianotasi!")
else:
    current_query_idx = [0]

    def show_next_query():
        clear_output(wait=True)
        if current_query_idx[0] >= len(remaining_queries):
            print("🎉 Semua query sudah dianotasi!")
            return

        query = remaining_queries[current_query_idx[0]]
        doc_indices = results_all[query][METHOD_TO_ANNOTATE]

        print(f"━"*60)
        print(f"📋 Query {current_query_idx[0]+1}/{len(remaining_queries)}: '{query}'")
        print(f"━"*60)
        print(f"\nAnotasi 10 dokumen di bawah (1=Relevan, 0=Tidak Relevan):\n")

        query_annotations = []
        doc_display = df.iloc[doc_indices][['Place_Name', 'City', 'Category', 'Description']].copy()

        for rank, (_, row) in enumerate(doc_display.iterrows(), 1):
            desc_snippet = str(row['Description'])[:100] + '...'
            print(f"  [{rank:2d}] {row['Place_Name']} ({row['City']}) - {row['Category']}")
            print(f"       {desc_snippet}\n")

        # Widget untuk input anotasi
        annotation_inputs = [
            widgets.ToggleButtons(
                options=['Relevan (1)', 'Tidak Relevan (0)'],
                value='Tidak Relevan (0)',
                description=f'Doc {i+1}:',
                style={'description_width': '60px'}
            )
            for i in range(len(doc_indices))
        ]

        save_btn = widgets.Button(description="Simpan & Lanjut →", button_style='success')

        def on_save(b):
            query_annot = [1 if inp.value == 'Relevan (1)' else 0 for inp in annotation_inputs]
            annotations[query] = query_annot
            with open(ANNOTATION_FILE, 'w') as f:
                json.dump(annotations, f, indent=2)
            current_query_idx[0] += 1
            show_next_query()

        save_btn.on_click(on_save)
        display(widgets.VBox(annotation_inputs + [save_btn]))

    show_next_query()

# Hitung Semua Metrik Evaluasi

In [ ]:
# Load annotations
with open(ANNOTATION_FILE, 'r') as f:
    annotations = json.load(f)

annotated_queries = list(annotations.keys())
print(f"Total query teranotasi: {len(annotated_queries)}")

# Evaluasi per metode
methods = ['hybrid', 'tfidf', 'bm25', 'dense']
K = 10

eval_results = {}
for method in methods:
    relevances_per_query = {}
    total_relevant_per_query = {}

    for query in annotated_queries:
        rels = annotations[query]
        relevances_per_query[query] = rels
        total_relevant_per_query[query] = max(sum(rels), 1)  # min 1 untuk hindari div/0

    metrics = evaluate_all_metrics(relevances_per_query, total_relevant_per_query, k=K)
    eval_results[method] = metrics
    print(f"\n📊 {method.upper():8s}: {metrics}")

# Tampilkan sebagai tabel
eval_df = pd.DataFrame(eval_results).T.round(4)
print(f"\n{'='*60}")
print("  RINGKASAN EVALUASI SEMUA METODE")
print(f"{'='*60}")
display(eval_df.style.highlight_max(color='lightgreen').highlight_min(color='#ffcccc'))

# Inter-Annotator Agreement (Cohen's Kappa)

In [ ]:
ANNOTATOR_2_FILE = f'{EVAL_DIR}/annotations_annotator_2.json'

if os.path.exists(ANNOTATOR_2_FILE):
    with open(ANNOTATOR_2_FILE, 'r') as f:
        annotations_2 = json.load(f)

    common_queries = [q for q in annotated_queries if q in annotations_2]
    print(f"Query yang dianotasi keduanya: {len(common_queries)}")

    # Flatten semua anotasi
    flat_a = []
    flat_b = []
    for q in common_queries:
        flat_a.extend(annotations[q])
        flat_b.extend(annotations_2[q])

    kappa = cohens_kappa(flat_a, flat_b)
    agree_pct = sum(a == b for a, b in zip(flat_a, flat_b)) / len(flat_a) * 100

    print(f"\n=== Inter-Annotator Agreement ===")
    print(f"Cohen's Kappa    : {kappa:.4f}")
    print(f"% Kesepakatan    : {agree_pct:.1f}%")

    # Interpretasi
    if kappa >= 0.8:
        interp = "Hampir sempurna (Almost Perfect)"
    elif kappa >= 0.6:
        interp = "Substansial (Substantial)"
    elif kappa >= 0.4:
        interp = "Cukup (Moderate)"
    elif kappa >= 0.2:
        interp = "Lemah (Fair)"
    else:
        interp = "Buruk (Poor)"
    print(f"Interpretasi     : {interp}")
else:
    print("⚠️  File anotator ke-2 belum ada.")
    print(f"   Buat anotasi ke-2 dengan ANNOTATOR_NAME = 'annotator_2'")

# Visualisasi Hasil Evaluasi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Perbandingan Performa Metode Retrieval', fontsize=14, fontweight='bold')

# Bar chart semua metrik
eval_df.plot(kind='bar', ax=axes[0], colormap='Set2', edgecolor='black')
axes[0].set_title('Semua Metrik per Metode')
axes[0].set_xlabel('Metode')
axes[0].set_ylabel('Skor')
axes[0].legend(loc='lower right', fontsize=8)
axes[0].tick_params(axis='x', rotation=0)
axes[0].set_ylim(0, 1.0)
for bar in axes[0].patches:
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f'{bar.get_height():.3f}',
        ha='center', va='bottom', fontsize=7
    )

# Radar chart
from matplotlib.patches import FancyArrowPatch
metrics_names = list(eval_df.columns)
N = len(metrics_names)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
axes[1].set_facecolor('#f8f8f8')

for i, (method, row) in enumerate(eval_df.iterrows()):
    values = row.tolist() + [row.tolist()[0]]
    axes[1].plot(angles, values, 'o-', linewidth=2, label=method.upper(), color=colors[i])
    axes[1].fill(angles, values, alpha=0.1, color=colors[i])

axes[1] = plt.subplot(122, polar=True)
axes[1].set_xticks(angles[:-1])
axes[1].set_xticklabels(metrics_names, fontsize=9)
axes[1].set_title('Radar Chart Perbandingan')
axes[1].set_ylim(0, 1)

for i, (method, row) in enumerate(eval_df.iterrows()):
    values = row.tolist() + [row.tolist()[0]]
    axes[1].plot(angles, values, 'o-', linewidth=2, label=method.upper(), color=colors[i])
    axes[1].fill(angles, values, alpha=0.1, color=colors[i])

axes[1].legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.savefig(f'{EVAL_DIR}/evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Grafik disimpan di data/evaluation_results.png")